# A4 — Estudo de caso: triagem de COVID-19 em raio-X com augmentation sintética por cGAN

**Como executar:** Google Colab com GPU T4 (*Ambiente de execução > Alterar o tipo de ambiente de
execução > T4 GPU*) e *Executar tudo*. A 4.2 (tráfego urbano) é só discussão escrita, no fim.

| | Execução completa |
|---|---|
| Tempo estimado de execução | [preencher após a execução final] |
| Pico de VRAM alocada (reservada) | [preencher] |
| Pico de RAM do processo | [preencher] |

> [Data e forma da medição, como na A1–A3.]

## Setup

Clone do código (`src/`), pasta de saída, imports e configuração. O código dos módulos está no
repositório; o notebook só orquestra.

In [ ]:
import os
import subprocess
import time

_T_INICIO_NOTEBOOK = time.perf_counter()  # tempo total do notebook (R15)
TEMPOS = []  # registro das etapas cronometradas

REPO_URL = "https://github.com/anacaetano02/Deep-Learning-Visao-Computacional.git"
SUBPASTA = "A4_estudo_caso_raio_x"
REPO_DIR = "/content/repo"
# Versão do código: na entrega, a tag da A4 (ex.: "a4-entrega"). None = branch principal.
GIT_REF = None

PROJECT_DIR = os.path.join(REPO_DIR, SUBPASTA)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # --filter=blob:none + --sparse: baixa só os arquivos da pasta da A4, não o repositório inteiro
    ref = ["--branch", GIT_REF] if GIT_REF else []
    subprocess.run(["git", "clone", "--filter=blob:none", "--sparse", "--depth", "1", *ref, REPO_URL, REPO_DIR],
                   check=True)
    subprocess.run(["git", "sparse-checkout", "set", SUBPASTA], cwd=REPO_DIR, check=True)
else:
    # Já clonado nesta sessão (talvez por outro notebook): garante a pasta da A4 no sparse checkout
    subprocess.run(["git", "sparse-checkout", "add", SUBPASTA], cwd=REPO_DIR, check=True)
    if GIT_REF:
        # Garante a versão fixada (não puxa commits novos)
        subprocess.run(["git", "fetch", "--depth", "1", "origin", "tag", GIT_REF], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "checkout", GIT_REF], cwd=REPO_DIR, check=True)
    else:
        print(f"Repositório já clonado em '{REPO_DIR}': buscando mudanças mais recentes...")
        subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

TEMPOS.append({"etapa": "clone/pull do repositório", "segundos": round(time.perf_counter() - _T_INICIO_NOTEBOOK, 1)})

In [ ]:
import importlib, sys

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Ao reexecutar o notebook na mesma sessão depois de atualizar o código, recarrega os módulos src.*
# já importados. Num runtime novo não há nada para recarregar e a célula não faz nada.
for nome in sorted(m for m in sys.modules if m == "src" or m.startswith("src.")):
    importlib.reload(sys.modules[nome])

In [ ]:
# Onde salvar os artefatos desta execução (split, checkpoints da GAN e dos classificadores, tabelas, figuras).
# * False (padrão): disco local do Colab, em /content (apagado quando o runtime é desconectado).
# * True: Google Drive (pede autorização no início).
from pathlib import Path

SALVAR_NO_DRIVE = False

if SALVAR_NO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/deep_learning_visao_computacional/A4_estudo_caso_raio_x")
else:
    BASE_DIR = Path("/content/A4_estudo_caso_raio_x_saida")

DIR_DADOS = BASE_DIR / "data"
DIR_OUTPUTS = BASE_DIR / "outputs"
DIR_REPORT_ASSETS = DIR_OUTPUTS / "report_assets"   # tabelas e figuras que vão para o relatório

for d in (DIR_DADOS, DIR_REPORT_ASSETS):
    d.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR = {BASE_DIR} (Drive: {SALVAR_NO_DRIVE})")

In [ ]:
# Configuração do experimento (cada escolha é justificada nos markdowns das seções)
SEED = 42
CLASSES = ["Normal", "Viral Pneumonia", "COVID"]   # Lung Opacity fica fora (não faz parte do caso)
# Escassez do projeto legado só no TREINO (840 / 240 / 120 = 70% / 20% / 10%); validação e teste reais,
# maiores, sorteados do restante. O teste com 300 COVID faz cada erro valer ~0,3 ponto de recall.
N_TREINO = {"Normal": 840, "Viral Pneumonia": 240, "COVID": 120}
N_VALIDACAO = {"Normal": 300, "Viral Pneumonia": 150, "COVID": 150}
N_TESTE = {"Normal": 600, "Viral Pneumonia": 300, "COVID": 300}
SEEDS_CLASSIFICADOR = [0, 1, 2]   # 3 treinos por condição (com × sem imagens geradas)

## 4.1 — Diagnóstico do projeto legado (R1, R2)

**O projeto:** 1.200 raio-X de tórax em 3 classes (Normal 840, Pneumonia 240, COVID-19 120); split 80/20
sem estratificação; ResNet-18 sem pré-treino; SGD com learning rate fixo de 0,01 por 15 épocas; sem
augmentation; avaliação só pela accuracy global. Resultado: 93% de accuracy no treino e 61% na validação,
e o time clínico relata que o modelo "ignora casos positivos".

**Um número resume o problema:** um modelo que respondesse sempre "Normal" teria **70% de accuracy**
(840/1.200), mais do que os 61% do modelo legado, e recall zero de COVID-19. Ou seja, o modelo é pior que
o trivial na métrica que o próprio projeto escolheu, e essa métrica nem enxerga o erro que importa.

Em triagem, o erro mais caro é o **falso negativo**: um paciente com COVID-19 classificado como Normal
deixa de ser isolado e tratado. O falso positivo custa um exame confirmatório (RT-PCR). Por isso, o impacto
clínico de cada problema abaixo é medido principalmente pelo que ele faz com a **sensibilidade (recall)
de COVID-19**.

| # | Problema técnico | Evidência no projeto | Impacto clínico esperado |
|---|---|---|---|
| 1 | **Desbalanceamento severo sem tratamento** (70% / 20% / 10%): sem pesos de classe, sem oversampling, sem ajuste de limiar | 120 COVID-19 contra 840 Normal; a cross-entropy padrão é dominada pela classe Normal | O modelo aprende que responder "Normal" minimiza a perda: **falsos negativos de COVID-19** (o "ignora casos positivos" relatado), com pacientes infectados liberados sem isolamento nem tratamento |
| 2 | **Métrica inadequada: só accuracy global** | 61% de validação, abaixo do baseline "sempre Normal" (70%); nenhum recall por classe | O problema do item 1 fica **invisível** para a equipe: o modelo poderia ser adotado sem que ninguém soubesse quantos casos de COVID-19 ele perde. Em triagem, a métrica principal deveria ser o recall (sensibilidade) de COVID-19, com a especificidade ao lado |
| 3 | **Split 80/20 sem estratificação e sem conjunto de teste** | Na validação, ~24 imagens de COVID-19 esperadas, podendo ser bem menos; a mesma validação escolhe o modelo e mede o desempenho | Uma estimativa de desempenho **instável e otimista**: com ~24 casos, cada erro muda o recall em ~4 pontos, e o número reportado não representa o que acontecerá com pacientes novos. Também não há garantia de separação por paciente (várias imagens do mesmo paciente em treino e validação inflariam o resultado) |
| 4 | **ResNet-18 sem pré-treino** (~11 milhões de parâmetros) para ~960 imagens de treino | Gap de **93% × 61%** entre treino e validação: overfitting | O modelo memoriza as imagens de treino em vez de aprender padrões de doença, e **falha em pacientes novos**. O pré-treino (ImageNet ou, melhor, em raio-X) daria features genéricas de bordas e texturas com muito menos dados |
| 5 | **Sem data augmentation** | Só 96 imagens de COVID-19 no treino (80% de 120), todas vistas da mesma forma a cada época | Reforça o overfitting do item 4 e torna o modelo **frágil a variações reais** (posição do paciente, contraste, equipamento), justamente na classe com menos exemplos |
| 6 | **Otimização sem controle**: SGD com lr fixo de 0,01, 15 épocas, sem scheduler, sem early stopping e sem seleção do melhor modelo pela validação | O modelo entregue é o da última época, seja ela boa ou não | Resultado **arbitrário e não reprodutível**: outro treino pode dar outro modelo; sem um checkpoint escolhido por um critério clínico (recall na validação), não há como garantir o desempenho do que é implantado |
| 7 | **Risco de atalhos (shortcut learning) e viés de fonte** | Em bases públicas, as imagens de COVID-19 costumam vir de hospitais e equipamentos diferentes das demais (marcações, bordas, contraste) | O modelo pode aprender **a fonte da imagem, e não a doença**, e falhar ao ser usado em outro hospital: os falsos negativos aparecem justamente onde o sistema seria implantado |
| 8 | **Sem limiar de decisão nem calibração** | Decisão por argmax, sem ajustar o limiar da classe COVID-19 | Em triagem, o limiar deveria ser escolhido para atingir uma sensibilidade alvo (ex.: ≥ 95%), aceitando mais falsos positivos. Sem isso, o equilíbrio entre falso negativo e falso positivo é decidido pelo acaso do treino, e não por critério clínico |

Os problemas se reforçam: o desbalanceamento (1) e a ausência de augmentation (5) levam o modelo a
ignorar COVID-19; o modelo sem pré-treino (4) e a otimização sem controle (6) causam o overfitting; e a
métrica (2) e o split (3) impedem que qualquer um desses problemas seja percebido antes da implantação. O
plano de melhoria (R11, R12) trata cada um deles.

## Dados: COVID-19 Radiography Database (Kaggle)

[Origem, versão, licença e citação; estrutura; quantas imagens por classe; o que fica de fora.]

In [ ]:
import kagglehub

from src.dados import ASSINATURA_DATASET, N_ARQUIVOS_DATASET, VERSAO_DATASET, impressao_digital

# Download com a versão fixada. No Colab, o kagglehub pode servir o cache do próprio Colab
# (/kaggle/input/...), cujo caminho não mostra a versão: quem garante que os dados são os mesmos é a
# impressão digital (nº de arquivos + hash da lista caminho|tamanho).
_t0 = time.perf_counter()
DIR_DATASET = Path(kagglehub.dataset_download(
    f"tawsifurrahman/covid19-radiography-database/versions/{VERSAO_DATASET}"))
TEMPOS.append({"etapa": "download do dataset", "segundos": round(time.perf_counter() - _t0, 1)})
print(DIR_DATASET)

n_arquivos, mb, assinatura = impressao_digital(DIR_DATASET)
assert (n_arquivos, assinatura) == (N_ARQUIVOS_DATASET, ASSINATURA_DATASET), "O conteúdo do dataset mudou"
print(f"{n_arquivos} arquivos, {mb} MB, assinatura {assinatura}")

In [ ]:
from collections import Counter

import pandas as pd

from src.dados import PASTA_RAIZ, arvore, carregar_imagem, ler_metadados, listar_imagens, remover_duplicatas

arvore(DIR_DATASET)
print("Extensões:", Counter(p.suffix.lower() for p in DIR_DATASET.rglob("*") if p.is_file()).most_common())

# Imagens das 3 classes do caso (as máscaras de pulmão e a classe Lung Opacity não são usadas)
imagens_brutas = listar_imagens(DIR_DATASET, CLASSES)
print(f"{len(imagens_brutas)} imagens:", imagens_brutas["classe"].value_counts().reindex(CLASSES).to_dict())

# Fonte de cada imagem (URL dos metadados de cada classe)
metadados = ler_metadados(DIR_DATASET, CLASSES)
imagens_brutas = imagens_brutas.merge(metadados[["classe", "nome", "url", "fonte"]], on=["classe", "nome"], how="left")
sem_fonte = imagens_brutas["fonte"].isna().sum()
print(f"Imagens sem linha nos metadados: {sem_fonte}")

# Duplicatas exatas (md5): mesma classe -> fica a primeira; classes diferentes -> conflito, saem todas.
# Feito ANTES do split, para uma cópia não cair no treino e a outra no teste.
imagens, duplicadas, conflitos = remover_duplicatas(imagens_brutas)
print(f"Duplicatas exatas: {len(duplicadas)} arquivos em {duplicadas['md5'].nunique()} grupos; "
      f"conflitos de classe: {len(conflitos)}")
if len(duplicadas):
    display(duplicadas.groupby("md5").agg(classe=("classe", "first"), arquivos=("arquivo", list)).reset_index(drop=True).head(20))
imagens["caminho"] = [str(DIR_DATASET / c) for c in imagens["caminho_rel"]]
print(f"Imagens após remover duplicatas: {len(imagens)}")

In [ ]:
import matplotlib.pyplot as plt

# EDA: o que importa para o experimento e para a análise crítica
print("Imagens por classe:", imagens["classe"].value_counts().reindex(CLASSES).to_dict())
print("Tamanho (px) e modo de cor por classe:")
display(imagens.groupby("classe").agg(largura_min=("largura", "min"), largura_max=("largura", "max"),
                                       altura_min=("altura", "min"), altura_max=("altura", "max"),
                                       modos=("modo", lambda m: dict(Counter(m)))).reindex(CLASSES))

# Fonte × classe: se cada classe vem de uma origem diferente, o classificador pode aprender a ORIGEM
# (hospital, equipamento, faixa etária) em vez da doença (shortcut learning)
fonte_classe = pd.crosstab(imagens["fonte"], imagens["classe"])[CLASSES]
display(fonte_classe)
fonte_classe.to_csv(DIR_REPORT_ASSETS / "fonte_por_classe.csv")

fig, ax = plt.subplots(figsize=(7, 3))
imagens["classe"].value_counts().reindex(CLASSES).plot.bar(ax=ax)
ax.set(title="Imagens por classe (dataset inteiro)", xlabel="", ylabel="imagens")
plt.tight_layout()
fig.savefig(DIR_REPORT_ASSETS / "imagens_por_classe.png", dpi=150)
plt.show()

# 6 exemplos sorteados por classe, com a fonte
N_EXEMPLOS = 6
exemplos = imagens.groupby("classe").sample(n=N_EXEMPLOS, random_state=SEED)
fig, eixos = plt.subplots(len(CLASSES), N_EXEMPLOS, figsize=(2.5 * N_EXEMPLOS, 2.8 * len(CLASSES)))
for i, classe in enumerate(CLASSES):
    for k, (_, r) in enumerate(exemplos[exemplos["classe"] == classe].iterrows()):
        ax = eixos[i, k]
        ax.imshow(carregar_imagem(r["caminho"]), cmap="gray")
        ax.set_title(f"{r['arquivo']}\n{r['fonte']}", fontsize=6)
        ax.set_xticks([]); ax.set_yticks([])
    eixos[i, 0].set_ylabel(classe, fontsize=11, fontweight="bold")
plt.tight_layout()
fig.savefig(DIR_REPORT_ASSETS / "exemplos_por_classe.png", dpi=110)
plt.show()

**Desenho do split.** [Escassez do legado só no treino (840/240/120); validação e teste reais e maiores,
do restante do dataset; por que o teste precisa ser maior (recall de COVID mensurável); sem
sobreposição entre splits.]

In [ ]:
from src.dados import SPLIT_TESTE, SPLIT_TREINO, SPLIT_VALIDACAO, dividir_com_escassez

# Split com escassez só no treino: teste e validação sorteados primeiro (quantidades fixas por classe),
# treino do restante. As imagens que sobram não são usadas.
imagens = dividir_com_escassez(imagens, N_TREINO, N_VALIDACAO, N_TESTE, SEED)
ORDEM_SPLITS = [SPLIT_TREINO, SPLIT_VALIDACAO, SPLIT_TESTE]
usadas = imagens[imagens["split"].notna()].reset_index(drop=True)

# Sem vazamento: nenhum conteúdo (md5) em mais de um split
assert (usadas.groupby("md5")["split"].nunique() == 1).all(), "Mesma imagem em mais de um split"
tabela_split = pd.crosstab(usadas["classe"], usadas["split"]).reindex(CLASSES)[ORDEM_SPLITS]
tabela_split.loc["total"] = tabela_split.sum()
display(tabela_split)
for s, esperado in ((SPLIT_TREINO, N_TREINO), (SPLIT_VALIDACAO, N_VALIDACAO), (SPLIT_TESTE, N_TESTE)):
    assert tabela_split[s].drop("total").to_dict() == esperado, (s, tabela_split[s].to_dict())
print("Fontes no treino de COVID:", usadas[(usadas["split"] == SPLIT_TREINO) & (usadas["classe"] == "COVID")]["fonte"].value_counts().to_dict())

# Split salvo para reprodutibilidade (como na A1 e na A3). Se a versão do repositório existir, o split
# recalculado tem que ser idêntico a ela.
COLUNAS_SPLIT = ["caminho_rel", "classe", "md5", "fonte", "split"]
usadas[COLUNAS_SPLIT].to_csv(DIR_REPORT_ASSETS / "split_imagens.csv", index=False)
SPLIT_VERSIONADO = Path(PROJECT_DIR) / "split_imagens.csv"
if SPLIT_VERSIONADO.exists():
    versionado = pd.read_csv(SPLIT_VERSIONADO)
    recalculado = usadas[COLUNAS_SPLIT].sort_values("caminho_rel").reset_index(drop=True)
    pd.testing.assert_frame_equal(recalculado, versionado.sort_values("caminho_rel").reset_index(drop=True))
    print("Split versionado confere com o recalculado.")
else:
    print(f"Split salvo em {DIR_REPORT_ASSETS / 'split_imagens.csv'} (copiar para {SUBPASTA}/ e versionar).")

## Classificador: ResNet-18 pré-treinada (R8)

[Por que a versão corrigida (pré-treinada) e não a do legado; entrada (tons de cinza → 3 canais,
resolução, normalização); configuração única usada nos treinos com e sem imagens geradas.]

In [ ]:
# [Dataset/DataLoader; augmentation clássica leve (se usada, igual nas duas condições)]

In [ ]:
# [função de treino do classificador (config fixa, seleção pela validação) e de avaliação (recall por classe)]

In [ ]:
# [treino SEM imagens geradas, para cada seed (R8)]

## cGAN para gerar raio-X de COVID-19 (R3, R4)

[Por que cGAN (e não CycleGAN); arquitetura (DCGAN condicional), resolução, dados de treino da GAN
(só o treino), custo na T4.]

In [ ]:
# [gerador e discriminador condicionais]

### Training loop adversarial (R5)

[Perdas, alternância D/G, otimizadores, o que é monitorado.]

In [ ]:
# [treino da cGAN com monitoramento: losses de D e G, D(x), D(G(z)), grade de amostras por época]

### Instabilidade: diagnóstico e mitigação (R6, R7)

[Sinais de mode collapse ou divergência na 1ª versão; estratégia de mitigação (ex.: label smoothing,
spectral norm, TTUR) e evidência de melhoria.]

In [ ]:
# [diagnóstico: curvas, diversidade das amostras, comparação antes × depois da mitigação]

In [ ]:
# [qualidade das imagens geradas: grade, diversidade, checagem de memorização (vizinho mais próximo no treino)]

## Classificador COM imagens geradas e comparação (R9, R10)

[Quantas imagens sintéticas de COVID entram no treino e por quê; mesma configuração do treino sem GAN.]

In [ ]:
# [treino COM imagens geradas, para cada seed (R9)]

In [ ]:
# [comparação: recall de COVID (média ± desvio entre seeds, IC), demais métricas, matrizes de confusão (R10)]

**Leitura da comparação (R10).** [O que muda no recall de COVID; se a diferença é maior que a variação
entre seeds; custo em outras classes; limitações.]

## Plano de melhoria integrado (R11, R12)

[Plano que endereça todos os problemas do diagnóstico: modelo, métrica, augmentation sintética e
critério de adoção clínica.]

## 4.2 — Estudo de caso: classificação de fluxo de tráfego (R13–R16)

[≥ 4 problemas técnicos/metodológicos; impactos em produção (chuva, noite, ângulos); riscos do transfer
learning de ImageNet para fluxo de tráfego; abordagem proposta para cada problema.]

## Apêndice: versões e tempo/memória (R17)

In [ ]:
# [versões das bibliotecas e resumo de tempo e memória, como na A3]